# E007 — Cold start and graduation

**Hypotheses:** H10, H8 ([docs/hypotheses.md](../../docs/hypotheses.md))

**Question:** What signal exists before an entity's first click, and when should a character get its own parameters?

**Method:** Permutation importance on cold slices of the test days; method-of-moments Beta priors per genre x tier with binomial noise removed; a causal per-character logit correction from strictly earlier impressions, evaluated by earlier-impression bucket.

In [1]:
from IPython.display import Markdown

from charade.analysis.experiment import setup

ctx = setup()
f"mode: {'smoke (fixture, tiny model)' if ctx.smoke else 'full data'}"

'mode: full data'

In [2]:
from charade.analysis import coldstart
from charade.analysis.experiment import ensure_bundle

ensure_bundle(ctx)
coldstart.run(ctx.settings, ctx.data_dir, ctx.reports / "coldstart")
Markdown((ctx.reports / "coldstart" / "coldstart.md").read_text())

# Cold start (generated by `uv run poe coldstart`)

Cold-character test rows: 1,478; new-user test rows: 97,843.

## Permutation importance, characters unseen in training (test)

| feature | delta_ne |
|---|---|
| site_id | 0.0323 |
| genre | 0.0269 |
| app_id | 0.0258 |
| app_category | 0.0081 |
| banner_pos | 0.0065 |
| user_seen | 0.0063 |
| device_model | 0.0048 |
| site_category | 0.0047 |
| site_domain | 0.0038 |
| C21 | 0.0027 |
| C19 | 0.0024 |
| safety_tier | 0.0023 |

## Permutation importance, users with no earlier impressions (test)

| feature | delta_ne |
|---|---|
| genre | 0.0381 |
| app_id | 0.0360 |
| site_id | 0.0310 |
| app_category | 0.0127 |
| site_domain | 0.0083 |
| device_model | 0.0072 |
| banner_pos | 0.0061 |
| safety_tier | 0.0047 |
| C14 | 0.0032 |
| site_category | 0.0023 |
| C21 | 0.0018 |
| device_id_real | 0.0014 |

## Beta prior per genre x tier (training characters with >= 50 impressions)

| genre | safety_tier | characters | mean | true_sd | prior_strength |
|---|---|---|---|---|---|
| anime | mature | 36 | 0.2023 | 0.0010 | 161380.7596 |
| anime | sfw | 139 | 0.1772 | 0.0083 | 2134.3269 |
| anime | suggestive | 62 | 0.1787 | 0.0074 | 2711.8198 |
| comedic | mature | 48 | 0.1998 | 0.0109 | 1339.5396 |
| comedic | sfw | 141 | 0.1755 | 0.0010 | 144675.3465 |
| comedic | suggestive | 71 | 0.1817 | 0.0187 | 424.9711 |
| fantasy | mature | 37 | 0.1928 | 0.0010 | 155631.0997 |
| fantasy | sfw | 132 | 0.1698 | 0.0010 | 140979.8347 |
| fantasy | suggestive | 70 | 0.1807 | 0.0010 | 148075.4472 |
| historical | mature | 44 | 0.1993 | 0.0010 | 159552.6372 |
| historical | sfw | 155 | 0.1748 | 0.0128 | 879.1487 |
| historical | suggestive | 57 | 0.1716 | 0.0010 | 142154.0824 |
| horror | mature | 45 | 0.2484 | 0.0010 | 186718.3776 |
| horror | sfw | 137 | 0.2239 | 0.0010 | 173761.8396 |
| horror | suggestive | 69 | 0.2237 | 0.0265 | 246.8961 |
| mentor | mature | 33 | 0.1595 | 0.0010 | 134068.1566 |
| mentor | sfw | 161 | 0.1456 | 0.0010 | 124418.0203 |
| mentor | suggestive | 73 | 0.1515 | 0.0094 | 1457.8021 |
| mystery | mature | 43 | 0.1921 | 0.0216 | 331.7189 |
| mystery | sfw | 142 | 0.1757 | 0.0010 | 144860.8420 |
| mystery | suggestive | 65 | 0.1762 | 0.0010 | 145139.7631 |
| romance | mature | 40 | 0.2644 | 0.0146 | 911.7304 |
| romance | sfw | 147 | 0.2248 | 0.0135 | 949.5028 |
| romance | suggestive | 62 | 0.2228 | 0.0010 | 173142.8408 |
| sci | mature | 27 | 0.1924 | 0.0010 | 155367.6894 |
| sci | sfw | 148 | 0.1746 | 0.0010 | 144113.3845 |
| sci | suggestive | 53 | 0.1748 | 0.0010 | 144217.1809 |
| slice | mature | 38 | 0.2100 | 0.0049 | 6888.8807 |
| slice | sfw | 156 | 0.1720 | 0.0010 | 142416.4299 |
| slice | suggestive | 61 | 0.1737 | 0.0038 | 10117.6372 |

## Cold ads (test): creatives and campaigns never seen in training

| slice | rows | share | ne | pred_over_obs |
|---|---|---|---|---|
| all test rows | 127,406 | 1.0000 | 0.8855 | 1.0021 |
| creative unseen in training | 54,602 | 0.4286 | 0.8886 | 0.9464 |
| campaign unseen in training | 53,458 | 0.4196 | 0.8867 | 0.9485 |
| creative and campaign seen | 72,804 | 0.5714 | 0.8891 | 1.0340 |


## Reading
- Publisher surface and genre carry the pre-click signal for both cold characters and new users.
- Most genre x tier cells show no spread between characters beyond binomial noise.
- Unseen creatives rank like seen ones but are under-predicted; see the cold-ads table.

## Conclusion
This synthetic dataset shows no reliable gain from character-specific parameters (no detectable spread beyond genre x tier); unseen characters score NE 0.908 vs 0.885 for warm ones (n = 1,478, no CI), and unseen creatives (43 % of test) are under-predicted by about 5 %.

**Decision:** No character ID in the model; on fresh data, re-run the per-cell spread and the character-ID ablation before adding one.